In [7]:
!pip install numpy

In [9]:
!pip install face-recognition

In [11]:
!pip install opencv-python

  Using cached opencv_python-4.10.0.84-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-4.10.0.84-cp37-abi3-win_amd64.whl (38.8 MB)


In [1]:
import cv2
import face_recognition
import os
import time
import numpy as np
from datetime import datetime
import dlib


TIME_LIMIT_SECONDS = 86400  # 24 hours time limit if the name in csv file exceeds then it is deleated 
COOLDOWN_SECONDS = 60       # Prevent entries within 60 seconds
MATCH_THRESHOLD = 0.5       # Face match distance threshold

# Load images and the classs names
path = "Person_Images"
images = []
classNames = []
mylist = os.listdir(path)

for cl in mylist:
    curImg = cv2.imread(f'{path}/{cl}')
    if curImg is None:
        print(f"Warning: Could not read image {cl}. Skipping.")
        continue
    images.append(curImg)
    classNames.append(os.path.splitext(cl)[0].title())  # Capitalized names

# Encoding stored faces 
def findEncodings(images):
    encodeList = []
    for idx, img in enumerate(images):
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        encodings = face_recognition.face_encodings(img)
        if encodings:
            encodeList.append(encodings[0])
        else:
            print(f"No face found in image {classNames[idx]}. Skipping.")
    return encodeList

encoded_face_train = findEncodings(images)

# Clear old attendance in the Attendance.csv 
def clearOldAttendance():
    now = datetime.now()
    if os.path.exists('Attendance.csv'):
        with open('Attendance.csv', 'r') as f:
            lines = f.readlines()

        updated_lines = []
        for line in lines:
            parts = line.strip().split(',')
            if len(parts) == 3:
                name, entry_time, entry_date = [p.strip() for p in parts]
                try:
                    entry_datetime = datetime.strptime(f'{entry_time} {entry_date}', '%I:%M:%S %p %d-%B-%Y')
                    if (now - entry_datetime).total_seconds() < TIME_LIMIT_SECONDS:
                        updated_lines.append(line)
                except ValueError:
                    continue

        with open('Attendance.csv', 'w') as f:
            f.writelines(updated_lines)

# Track recent attendance to avoid rapid re-entries
recent_attendance = {}

# Mark attendance of the person with a cooldown time 
def markAttendance(name):
    now = datetime.now()
    current_time = now.strftime('%I:%M:%S %p')
    current_date = now.strftime('%d-%B-%Y')

    if name in recent_attendance:
        last_seen = recent_attendance[name]
        if (now - last_seen).total_seconds() < COOLDOWN_SECONDS:
            return

    recent_attendance[name] = now

    with open('Attendance.csv', 'a') as f:
        f.write(f'{name}, {current_time}, {current_date}\n')

# One-time cleanup on start
clearOldAttendance()

# Start webcam
cap = cv2.VideoCapture(0)
start_time = time.time()

while True:
    success, img = cap.read()
    if not success:
        break

    
    imgS = cv2.resize(img, None, fx=0.25, fy=0.25)
    imgS = cv2.cvtColor(imgS, cv2.COLOR_BGR2RGB)

    faces_in_frame = face_recognition.face_locations(imgS)
    encoded_faces = face_recognition.face_encodings(imgS, faces_in_frame)

    for encode_face, faceloc in zip(encoded_faces, faces_in_frame):
        faceDist = face_recognition.face_distance(encoded_face_train, encode_face)
        matchIndex = np.argmin(faceDist)

        name = "Unknown"
        if len(faceDist) > 0 and faceDist[matchIndex] < MATCH_THRESHOLD:
            name = classNames[matchIndex]

        y1, x2, y2, x1 = faceloc
        y1, x2, y2, x1 = [v * 4 for v in (y1, x2, y2, x1)]

        color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.rectangle(img, (x1, y2 - 35), (x2, y2), color, cv2.FILLED)
        cv2.putText(img, name, (x1 + 6, y2 - 6), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2)

        if name != "Unknown":
            markAttendance(name)

    cv2.imshow('Webcam', img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    if time.time() - start_time > 20:  # Auto-stop after 20 seconds
        break

cap.release()
cv2.destroyAllWindows()


# 